# Neural Network approach to function 5

In [1]:
import numpy as np
import tensorflow as tf
import pickle
import os

### Data preparation

In [2]:
input = np.load('./initial_inputs.npy')
output = np.load('./initial_outputs.npy')
print(input)
print(output)
#print(input.shape)
#print(input.shape[1])
func_dimensions = input.shape[1]
print('this function has ', func_dimensions, ' dimensions')

[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]]
[6.44434399e+01 1.83013796e+01 1.12939795e-01 4.21089813e+0

Additional data provided by weekly queries

In [3]:
additionalInputs = [[0.5, 0.5, 0.5, 0.5], [0.210195, 0.849014, 0.874271, 0.875537], [0.241981, 0.847887, 0.878175, 0.876911], [0.230056, 0.841143, 0.873217, 0.887638], [0.241981, 0.847887, 0.878175, 0.876911], [0.0, 0.24, 0.62, 1.0]]
additionalOutputs = [np.float64(32.0025), np.float64(1060.3898045443261), np.float64(1083.0108574790386), np.float64(1081.4793716878269), np.float64(1083.0108574790386), np.float64(321.73026824755283)]

input = np.append(input, additionalInputs, axis=0)
output = np.append(output, additionalOutputs)

print(input)
print(output)

[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.5        0.5        0.5        0.5       ]
 [0.210195   

# Neural network setup
Here we prepare a neural network to approach function 5

This requires 4 inputs and 1 output

We decide to apply 2 hidden layers of 8 nodes each

The activation function will be a tanh


In [4]:
# 1. Define the input data and target output for demonstration
x_train = np.array(input, dtype=np.float32)
y_train = np.array(output, dtype=np.float32).reshape(-1, 1)

# Normalize the output (CRITICAL for training)
y_mean = np.mean(y_train)
y_std = np.std(y_train)
y_train_normalized = (y_train - y_mean) / y_std

print(f"Output normalized: mean={y_mean:.2f}, std={y_std:.2f}")



Output normalized: mean=295.66, std=398.82


In [5]:
print(x_train)
print(y_train)
print(len(x_train))
print(len(y_train))

[[0.19144708 0.03819337 0.6074178  0.41458413]
 [0.7586529  0.53651774 0.6560004  0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129481]
 [0.7060508  0.53419197 0.26424333 0.48208755]
 [0.836478   0.19360964 0.6638927  0.7856489 ]
 [0.6834322  0.11866264 0.8290459  0.5675766 ]
 [0.5536215  0.66735    0.3238058  0.81486976]
 [0.35235626 0.32224154 0.11697937 0.47311252]
 [0.1537857  0.7293817  0.42259845 0.44307417]
 [0.46344227 0.6300245  0.10790645 0.9576439 ]
 [0.6774911  0.3585095  0.47959223 0.07288048]
 [0.5839734  0.14724265 0.34809747 0.42861465]
 [0.30688873 0.31687814 0.6226345  0.09539906]
 [0.5111418  0.817957   0.7287104  0.11235362]
 [0.43893337 0.7740918  0.3781671  0.9336962 ]
 [0.22418903 0.8464805  0.8794842  0.87851566]
 [0.72526175 0.4798705  0.08894684 0.7597602 ]
 [0.3554816  0.63961935 0.41761768 0.12260384]
 [0.11987922 0.8625403  0.64333135 0.8498038 ]
 [0.12688467 0.15342963 0.77016217 0.19051811]
 [0.5        0.5        0.5        0.5       ]
 [0.210195   

In [6]:
# Initialize weights with better initialization (Xavier/Glorot)
W1 = tf.Variable(tf.random.normal([5, 8], stddev=np.sqrt(2.0/5)), dtype=tf.float32)  # 4 inputs + 1 bias
W2 = tf.Variable(tf.random.normal([9, 8], stddev=np.sqrt(2.0/9)), dtype=tf.float32)  # 8 hidden + 1 bias
W3 = tf.Variable(tf.random.normal([8, 1], stddev=np.sqrt(2.0/8)), dtype=tf.float32)  # 8 hidden to 1 output
print(W1)
print(W2)
print(W3)


<tf.Variable 'Variable:0' shape=(5, 8) dtype=float32, numpy=
array([[-0.8522449 ,  0.02100893, -1.8258872 ,  0.22193004, -0.42486644,
         1.7371207 ,  0.10913581,  0.10417305],
       [-0.33329555, -0.30993858,  0.7754741 ,  0.8524086 , -0.01525033,
        -0.29874733,  0.4617136 , -0.89009964],
       [-0.45679992, -0.24368761, -0.38413012, -0.252068  , -1.0345495 ,
        -0.5455107 ,  0.42512742,  0.8182169 ],
       [ 0.05577606, -0.6957329 , -0.32628918, -0.9251804 ,  0.05255438,
        -0.43071905, -0.349936  , -0.6618562 ],
       [-1.0292648 , -0.34456468,  0.6260025 ,  0.4907431 ,  0.54754704,
        -0.2997415 ,  0.36925063,  0.766603  ]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(9, 8) dtype=float32, numpy=
array([[ 1.2007314e-01, -6.7962545e-01, -7.1228731e-01, -1.1020663e+00,
        -1.7016354e-01,  4.2705506e-01,  2.4537982e-01,  2.7814725e-01],
       [ 4.9970457e-01, -1.0462695e-01,  2.4719758e-01,  1.2101307e+00,
         2.7337459e-01,  2.0185317e-01,

In [7]:
@tf.function
def forward_pass(x_batch):
    # x_batch shape: (batch_size, 4)
    batch_size = tf.shape(x_batch)[0]
    bias = tf.ones((batch_size, 1), dtype=tf.float32)

    # First hidden layer
    x_with_bias = tf.concat([x_batch, bias], axis=1)  # (batch_size, 5)
    hidden1 = tf.nn.tanh(tf.matmul(x_with_bias, W1))  # (batch_size, 8)

    # Second hidden layer
    hidden1_with_bias = tf.concat([hidden1, bias], axis=1)  # (batch_size, 9)
    hidden2 = tf.nn.tanh(tf.matmul(hidden1_with_bias, W2))  # (batch_size, 8)

    # Output layer (LINEAR, no sigmoid!)
    output = tf.matmul(hidden2, W3)  # (batch_size, 1)

    return output


#print(forward_pass(x_train[0].reshape(4,-1)))

In [8]:
# Loss and optimizer
loss_fn = tf.keras.losses.MeanSquaredError()
optimizer = tf.optimizers.SGD(learning_rate=0.01)  # Lower learning rate
print(loss_fn)
print(optimizer)

<LossFunctionWrapper(<function mean_squared_error at 0x7b74496ac9a0>, kwargs={})>


In [10]:
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        y_pred = forward_pass(x)
        loss = loss_fn(y, y_pred)
    gradients = tape.gradient(loss, [W1, W2, W3])
    optimizer.apply_gradients(zip(gradients, [W1, W2, W3]))
    return loss


###END SOLUTION
#for i, v in enumerate(x_train):
#  print(train_step(x_train[i].reshape(4,-1), y_train[i]))
#for epoch in range(10):
#  print(train_step(x_train[0].reshape(4,-1), y_train[0].reshape(1,1)))
# Training loop
print("\nTraining...")
epochs = 50000
for epoch in range(epochs):
    loss = train_step(x_train, y_train_normalized)

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.numpy():.6f}")



Training...
Epoch 0, Loss: 0.012174
Epoch 500, Loss: 0.010651
Epoch 1000, Loss: 0.009414
Epoch 1500, Loss: 0.008407
Epoch 2000, Loss: 0.007588
Epoch 2500, Loss: 0.006920
Epoch 3000, Loss: 0.006373
Epoch 3500, Loss: 0.005921
Epoch 4000, Loss: 0.005542
Epoch 4500, Loss: 0.005221
Epoch 5000, Loss: 0.004945
Epoch 5500, Loss: 0.004702
Epoch 6000, Loss: 0.004486
Epoch 6500, Loss: 0.004292
Epoch 7000, Loss: 0.004114
Epoch 7500, Loss: 0.003948
Epoch 8000, Loss: 0.003794
Epoch 8500, Loss: 0.003647
Epoch 9000, Loss: 0.003508
Epoch 9500, Loss: 0.003374
Epoch 10000, Loss: 0.003245
Epoch 10500, Loss: 0.003119
Epoch 11000, Loss: 0.002997
Epoch 11500, Loss: 0.002877
Epoch 12000, Loss: 0.002761
Epoch 12500, Loss: 0.002646
Epoch 13000, Loss: 0.002534
Epoch 13500, Loss: 0.002425
Epoch 14000, Loss: 0.002317
Epoch 14500, Loss: 0.002212
Epoch 15000, Loss: 0.002110
Epoch 15500, Loss: 0.002010
Epoch 16000, Loss: 0.001913
Epoch 16500, Loss: 0.001820
Epoch 17000, Loss: 0.001729
Epoch 17500, Loss: 0.001641
Epo

In [11]:
# ============================================================================
# SAVE WEIGHTS FUNCTION
# ============================================================================
def save_weights(filepath='model_weights.pkl'):
    """
    Save the trained weights and normalization parameters to a file.

    Args:
        filepath: Path where to save the weights (default: 'model_weights.pkl')
    """
    weights_dict = {
        'W1': W1.numpy(),
        'W2': W2.numpy(),
        'W3': W3.numpy(),
        'y_mean': y_mean,
        'y_std': y_std
    }

    with open(filepath, 'wb') as f:
        pickle.dump(weights_dict, f)

    print(f"\n✓ Weights saved to '{filepath}'")


In [12]:
# ============================================================================
# LOAD WEIGHTS FUNCTION
# ============================================================================
def load_weights(filepath='model_weights.pkl'):
    """
    Load previously saved weights and normalization parameters.

    Args:
        filepath: Path to the saved weights file

    Returns:
        tuple: (y_mean, y_std) normalization parameters
    """
    global W1, W2, W3, y_mean, y_std

    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Weights file '{filepath}' not found!")

    with open(filepath, 'rb') as f:
        weights_dict = pickle.load(f)

    # Load weights into TensorFlow variables
    W1.assign(weights_dict['W1'])
    W2.assign(weights_dict['W2'])
    W3.assign(weights_dict['W3'])

    # Load normalization parameters
    y_mean = weights_dict['y_mean']
    y_std = weights_dict['y_std']

    print(f"\n✓ Weights loaded from '{filepath}'")
    return y_mean, y_std


In [13]:
# ============================================================================
# PREDICTION FUNCTION (for inference with loaded model)
# ============================================================================
def predict(x_input, y_mean, y_std):
    """
    Make predictions on new data using the loaded model.

    Args:
        x_input: Input data (numpy array of shape [n_samples, 4])
        y_mean: Mean used for normalization
        y_std: Standard deviation used for normalization

    Returns:
        Predictions in original scale
    """
    x_input = np.array(x_input, dtype=np.float32)
    if len(x_input.shape) == 1:
        x_input = x_input.reshape(1, -1)

    predictions_normalized = forward_pass(x_input).numpy()
    predictions = predictions_normalized * y_std + y_mean

    return predictions


In [14]:
# ============================================================================
# TRAINING MODE
# ============================================================================
TRAIN_MODE = True  # Set to False to skip training and only load weights

if TRAIN_MODE:
    print("\nTraining...")
    epochs = 50000
    for epoch in range(epochs):
        loss = train_step(x_train, y_train_normalized)

        if epoch % 500 == 0:
            print(f"Epoch {epoch}, Loss: {loss.numpy():.6f}")

    # Make predictions
    print("\n" + "="*70)
    print("Training Results:")
    print("="*70)
    predictions_normalized = forward_pass(x_train).numpy()
    predictions = predictions_normalized * y_std + y_mean

    print(f"{'Actual':<15} {'Predicted':<15} {'Error':<15}")
    print("-"*70)
    for i in range(len(y_train)):
        actual = y_train[i, 0]
        predicted = predictions[i, 0]
        error = abs(actual - predicted)
        print(f"{actual:<15.2f} {predicted:<15.2f} {error:<15.2f}")

    mse = np.mean((predictions - y_train) ** 2)
    mae = np.mean(np.abs(predictions - y_train))
    print("="*70)
    print(f"Mean Squared Error: {mse:.2f}")
    print(f"Mean Absolute Error: {mae:.2f}")

    # Save the trained weights
    save_weights('model_weights.pkl')

# ============================================================================
# INFERENCE MODE (Load weights and make predictions)
# ============================================================================
else:
    print("\n" + "="*70)
    print("INFERENCE MODE - Loading saved weights")
    print("="*70)

    # Load previously saved weights
    y_mean, y_std = load_weights('model_weights.pkl')

    # Make predictions on the training data (or new data)
    print("\nMaking predictions with loaded model...")
    predictions = predict(x_train, y_mean, y_std)

    print(f"\n{'Actual':<15} {'Predicted':<15} {'Error':<15}")
    print("-"*70)
    for i in range(len(y_train)):
        actual = y_train[i, 0]
        predicted = predictions[i, 0]
        error = abs(actual - predicted)
        print(f"{actual:<15.2f} {predicted:<15.2f} {error:<15.2f}")

    # Example: Make a prediction on a new sample
    print("\n" + "="*70)
    print("Example prediction on new data:")
    print("="*70)
    new_sample = np.array([[0.5, 0.5, 0.5, 0.5]])
    new_prediction = predict(new_sample, y_mean, y_std)
    print(f"Input: {new_sample[0]}")
    print(f"Predicted output: {new_prediction[0, 0]:.2f}")


Training...
Epoch 0, Loss: 0.000272
Epoch 500, Loss: 0.000270
Epoch 1000, Loss: 0.000268
Epoch 1500, Loss: 0.000266
Epoch 2000, Loss: 0.000265
Epoch 2500, Loss: 0.000263
Epoch 3000, Loss: 0.000261
Epoch 3500, Loss: 0.000259
Epoch 4000, Loss: 0.000258
Epoch 4500, Loss: 0.000256
Epoch 5000, Loss: 0.000254
Epoch 5500, Loss: 0.000253
Epoch 6000, Loss: 0.000251
Epoch 6500, Loss: 0.000250
Epoch 7000, Loss: 0.000248
Epoch 7500, Loss: 0.000247
Epoch 8000, Loss: 0.000245
Epoch 8500, Loss: 0.000244
Epoch 9000, Loss: 0.000242
Epoch 9500, Loss: 0.000241
Epoch 10000, Loss: 0.000240
Epoch 10500, Loss: 0.000238
Epoch 11000, Loss: 0.000237
Epoch 11500, Loss: 0.000236
Epoch 12000, Loss: 0.000234
Epoch 12500, Loss: 0.000233
Epoch 13000, Loss: 0.000232
Epoch 13500, Loss: 0.000231
Epoch 14000, Loss: 0.000229
Epoch 14500, Loss: 0.000228
Epoch 15000, Loss: 0.000227
Epoch 15500, Loss: 0.000226
Epoch 16000, Loss: 0.000225
Epoch 16500, Loss: 0.000224
Epoch 17000, Loss: 0.000223
Epoch 17500, Loss: 0.000221
Epo

In [15]:
# ============================================================================
# GRID SEARCH FOR MAXIMUM OUTPUT
# ============================================================================
print("\n" + "="*70)
print("Searching for maximum output on grid...")
print("="*70)

# Initialize grid for plots
vector_values = np.linspace(0, 1, 101)  # 101 values from 0 to 1

# Generalized version for any number of dimensions
grids = np.meshgrid(*([vector_values] * 4), indexing='ij')
x_grid = np.column_stack([grid.ravel() for grid in grids])

print(f"Grid size: {x_grid.shape[0]:,} points ({len(vector_values)}^4)")
print("Evaluating model on entire grid...")


Searching for maximum output on grid...
Grid size: 104,060,401 points (101^4)
Evaluating model on entire grid...


In [16]:
print(x_grid.shape)

(104060401, 4)


In [17]:

# Make predictions on the entire grid (may take a moment for 101^4 = ~104M points)
# Process in batches to avoid memory issues
batch_size = 100000
n_batches = int(np.ceil(len(x_grid) / batch_size))
all_predictions = []

for i in range(n_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(x_grid))
    batch_predictions = predict(x_grid[start_idx:end_idx], y_mean, y_std)
    all_predictions.append(batch_predictions)

    if (i + 1) % 10 == 0 or (i + 1) == n_batches:
        print(f"  Progress: {i+1}/{n_batches} batches processed")


  Progress: 10/1041 batches processed
  Progress: 20/1041 batches processed
  Progress: 30/1041 batches processed
  Progress: 40/1041 batches processed
  Progress: 50/1041 batches processed
  Progress: 60/1041 batches processed
  Progress: 70/1041 batches processed
  Progress: 80/1041 batches processed
  Progress: 90/1041 batches processed
  Progress: 100/1041 batches processed
  Progress: 110/1041 batches processed
  Progress: 120/1041 batches processed
  Progress: 130/1041 batches processed
  Progress: 140/1041 batches processed
  Progress: 150/1041 batches processed
  Progress: 160/1041 batches processed
  Progress: 170/1041 batches processed
  Progress: 180/1041 batches processed
  Progress: 190/1041 batches processed
  Progress: 200/1041 batches processed
  Progress: 210/1041 batches processed
  Progress: 220/1041 batches processed
  Progress: 230/1041 batches processed
  Progress: 240/1041 batches processed
  Progress: 250/1041 batches processed
  Progress: 260/1041 batches proce

In [18]:

# Concatenate all predictions
grid_predictions = np.concatenate(all_predictions, axis=0)

# Find the maximum
max_idx = np.argmax(grid_predictions)
max_input = x_grid[max_idx]
max_output = grid_predictions[max_idx, 0]

print("\n" + "="*70)
print("MAXIMUM OUTPUT FOUND:")
print("="*70)
print(f"Input values:  [{max_input[0]:.6f}, {max_input[1]:.6f}, {max_input[2]:.6f}, {max_input[3]:.6f}]")
print(f"Predicted output: {max_output:.2f}")
print("="*70)



MAXIMUM OUTPUT FOUND:
Input values:  [0.290000, 0.990000, 1.000000, 1.000000]
Predicted output: 1449.21


In [19]:
# Also show top 5 candidates
print("\nTop 5 inputs with highest predicted outputs:")
print("-"*70)
top_5_indices = np.argsort(grid_predictions[:, 0])[-5:][::-1]
for rank, idx in enumerate(top_5_indices, 1):
    inp = x_grid[idx]
    out = grid_predictions[idx, 0]
    print(f"{rank}. Input: [{inp[0]:.4f}, {inp[1]:.4f}, {inp[2]:.4f}, {inp[3]:.4f}] → Output: {out:.2f}")


Top 5 inputs with highest predicted outputs:
----------------------------------------------------------------------
1. Input: [0.2900, 0.9900, 1.0000, 1.0000] → Output: 1449.21
2. Input: [0.2900, 0.9800, 1.0000, 1.0000] → Output: 1449.19
3. Input: [0.2900, 1.0000, 1.0000, 1.0000] → Output: 1449.17
4. Input: [0.2900, 0.9700, 1.0000, 1.0000] → Output: 1449.11
5. Input: [0.3000, 1.0000, 1.0000, 1.0000] → Output: 1449.08
